J3 - Modélisation : régression

# Étape 3 — Modélisation & ACP (FAO DataLab)

## 🎯 Objectifs
- Construire un modèle de régression pour imputer les valeurs manquantes de sous‑nutrition.
- Utiliser les features : 
  - disponibilité calorique totale (kcal/habitant),
  - part de protéines animales,
  - part de céréales,
  - population.
- Évaluer le modèle avec :
  - R²,
  - RMSE,
  - validation croisée.
- Comparer à une baseline simple.
- Bonus : réaliser une ACP sur les indicateurs alimentaires.


In [4]:
# 1. Imports & chargement du dataset global

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from pathlib import Path

sns.set(style="whitegrid")

DATA_DIR = Path("../data")
df = pd.read_csv(DATA_DIR / "global_fao_dataset.csv")
# Renommer la colonne longue en français et garder un alias `Kcal_total` pour compatibilité

french_name = "Kcal_total_kcal_par_habitant_par_jour"
if "Kcal_total_kcal_per_capita_per_day" in df.columns:
    df = df.rename(columns={"Kcal_total_kcal_per_capita_per_day": french_name})

# Si la colonne `Kcal_total` n'existe pas encore mais que la version française existe, créer un alias
if "Kcal_total" not in df.columns and french_name in df.columns:
    df["Kcal_total"] = df[french_name]

# Sinon, si les composants existent, construire `Kcal_total` à partir de `Kcal_veg` + `Kcal_anim`
elif "Kcal_total" not in df.columns and {"Kcal_veg", "Kcal_anim"}.issubset(df.columns):
    df["Kcal_total"] = df["Kcal_veg"].fillna(0) + df["Kcal_anim"].fillna(0)


## 2. Préparation des features

Features retenues :
- `Kcal_total`
- `Part_proteines_animales`
- `Part_cereales` 
- `Population_millions`

Target :
- `Taux_sous_nutrition_pct`


In [5]:
# Calcul des parts

df["Part_animaux"] = df["Kcal_anim"] / df["Kcal_total"]
df["Part_veg"] = df["Kcal_veg"] / df["Kcal_total"]


# Selection des features

features = ["Kcal_total", "Part_animaux", "Population_millions"]
target = "Taux_sous_nutrition_pct"

df_model = df[features + [target]].dropna()


In [6]:
# 3.Split train/test

X = df_model[features]
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
